In [1]:
%load_ext autoreload
%autoreload 2
path_src = "../../src"
import sys
from pathlib import Path
import pandas as pd

sys.path.append(path_src)
import numpy as np
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from metrics import remove_by_size, root, label_files, size_files, meta_files, visualize_iom, add_size_based_fields

from froc_overlap_2 import FROCEvaluator
import os

from metrics_config import inf_appends, thresholds


In [2]:
max_fppi = 10.0
min_fppi = 0.0
fp_scale = "linear"
fppi_thrs = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
n_bootstraps = 10000
total_volumes = {
    "internal_test": 152,
    "ext": 138,
    "hospital": 38,
    "hospital140": 143,
    "cmha": 140,
    "internal_train": 1186,
    "cta_rsna_ane": 843,
}

In [13]:
iou_thr = 0.1
exp_base = "decoder_only_no_rec_input_edt"
inf_append = "50k"

values = []

datasets = ["cta_rsna_ane"]
modes = ["base", "1", "2", "3", "4", "5"]
for dataset in datasets:
    for mode in modes:
        # exp = exps[0]
        print("\n" + "---" * 10 + "\n")
        path_inf = "inference_" + inf_append
        print(f"Running iou_thr: {iou_thr} at {inf_append}")
        
        n_workers = 8
        path_preds = (
            root / f"results/{dataset}/{exp_base}/{path_inf}/predict_roi_jisoo_filtered.csv"
        )
        print(path_preds)
        if not os.path.isfile(path_preds):
            print(f"No data for checkpoint {inf_append}/partition {mode}")
            continue
        preds = pd.read_csv(path_preds)
        if mode == "0" or mode is None:
            # remove preds that have overlap with brain <= 0.5
            preds = preds[preds["overlap"] > 0.5]
        elif mode == "base":
            # no filtering
            preds = preds.copy()
        elif mode == "1":
            # remove preds that have overlap with enhanced brain <= 0.5
            preds = preds[preds["overlap_enhanced_brain"] > 0.5]
        elif mode == "2":
            # remove preds that have any overlap with vein
            preds = preds[preds["overlap_vein"] == 0]
        elif mode == "3":
            # remove preds that have more overlap with vein than artery
            preds = preds[
                (preds["overlap_vein"] <= preds["overlap_artery"])
            ]
        elif mode == "5":
            # remove preds that have more overlap with vein than artery
            # then keep those with overlap with enhanced brain > 0.5
            preds = preds[
                (preds["overlap_vein"] <= preds["overlap_artery"])
                & (preds["overlap_enhanced_brain"] > 0.5)
            ]
        elif mode == "4":
            # remove preds that have any overlap with vein
            # then keep those with overlap with enhanced brain > 0.5
            preds = preds[preds["overlap_vein"] == 0]
            preds = preds[preds["overlap_enhanced_brain"] > 0.5]
        sizes = None
        label_file = label_files[dataset]

        # label_file = "/projects/vig/Datasets/aneurysm/cta_datasets/hospital140/annotations.csv"
        if mode == "hospital":
            sizes = pd.read_json(size_files["hospital"])

        df_labels = pd.read_csv(label_file)
        df_labels = add_size_based_fields(df_labels)
        df_labels = df_labels.sort_values(by="seriesuid")
        #
        # df_labels = df_labels.sample(frac=1, random_state=5).reset_index(drop=True)
        # df_labels_small = df_labels[df_labels["size"] == "small"].sample(n=7)
        # df_labels_large = df_labels[df_labels["size"] == "large"].sample(n=3)
        # df_labels_med = df_labels[df_labels["size"] == "medium"].sample(n=23)
        # df_labels = pd.concat([df_labels_small, df_labels_large, df_labels_med])
        # df_labels.to_csv("../../labels/gt/hospital_crop_0.4_subsample.csv")
        # # 28 + 9 + 3
        # break
        # df_labels[["intersection_art", "intersection_vein", "avg_distance_art"]] = (
        #print("Len before", preds.shape[0])
        #preds = preds[preds["overlap"] > 0.5]
        #print("Len after:", preds.shape[0])
        if mode == "hospital":
            preds = remove_by_size(preds, sizes)
        vols = total_volumes[dataset]
        evaluator = FROCEvaluator(
            label_file=label_file,
            preds=preds,
            logger=None,
            iou_thr=iou_thr,
            out_dir=root
            / f"results/{dataset}/{exp_base}/wiwawe/iou{iou_thr:.1f}_froc_{inf_append}",
            max_fppi=max_fppi,
            fppi_thrs=fppi_thrs,
            min_fppi=min_fppi,
            n_bootstraps=n_bootstraps,
            n_workers=n_workers,
            fp_scale=fp_scale,
            use_world_xyz=False,
            exp_name=exp_base + "_" + inf_append,
            mode=dataset,
        )
        evaluator.evaluate()
        total_correct = 0
        total_correct_any = 0
        total_fp = 0
        total_fn = 0

        correct_per_case = []
        fp_per_case = []
        fn_per_case = []
        gt_per_case = []
        correct_dets = []
        fp_list = []
        # path_thrs = (
        #     root
        #     / f"results/{dataset}/{exp_base}/iou{iou_thr:.1f}_froc_{inf_append}/thres_aneurysm.npy"
        # )
        # thrs = np.load(path_thrs)
        # t = thrs[2]
        t = 0.8

        preds = preds[preds["probability"] > t]

        case_names = []

        for key in evaluator._match_results.keys():
            gt_per_case.append(len(df_labels[df_labels["seriesuid"] == key]))
            un_matched_gt = evaluator._match_results[key]["aneurysm"]["un_matched_gt"]
            all_un_matched_gt = evaluator._match_results[key]["aneurysm"][
                "all_un_matched_gt"
            ]
            filtered_umgt = [x for x in all_un_matched_gt if x[0] > t]
            un_matched_gt = (
                filtered_umgt[-1][-1] if len(filtered_umgt) > 0 else np.array([])
            )
            case_names.append(key)
            if len(un_matched_gt) > 1:
                un_matched_gt = un_matched_gt[0:-1]
                total_correct_any += np.sum(un_matched_gt == 0)
                correct_dets += list(un_matched_gt == 0)
                fn = np.sum(un_matched_gt)
            else:
                fn = len(df_labels[df_labels["seriesuid"] == key])
                correct_dets += [0] * len(df_labels[df_labels["seriesuid"] == key])
            fn_per_case.append(fn)
            scores = evaluator._match_results[key]["aneurysm"]["scores"].copy()
            gt = evaluator._match_results[key]["aneurysm"]["gts"].copy()
            gt[scores < t] = 0
            correct_per_case.append(np.sum(gt))
            fp_per_case.append(np.sum(gt[scores >= t] == 0))
            is_fp = list(gt[scores >= t] == 0)
            fp_list += is_fp
            # for res in evaluator._match_results[key]:

        total_fp = np.sum(fp_per_case)
        total_fn = np.sum(fn_per_case)
        total_correct = np.sum(correct_per_case)
        total_gt = np.sum(gt_per_case)
        df_labels["detected"] = np.array(correct_dets).astype(np.int8)

        df_labels["size"].value_counts()
        preds["is_fp"] = np.array(fp_list).astype(np.int8)
        healthy_case_indexes = np.where(np.array(gt_per_case) == 0)[0]
        healthy_fp = np.array(fp_per_case)[healthy_case_indexes]
        healthy_fp = healthy_fp[healthy_fp > 0]
        sick_case_indics = np.where(np.array(gt_per_case) > 0)[0]
        sick_fn = np.array(fn_per_case)[sick_case_indics]
        print(f"Dataset: {dataset}, Model: {exp_base}, Mode: {mode}")
        #print("Total cases:", vols)

        #print("Number of aneurysms:", total_gt)

        #print("Healthy cases:", len(healthy_case_indexes))
        print("Confidence threshold:", t)
        #print("Recall:", 100 * total_correct / total_gt)
        print("Total correct", total_correct)
        values.append(total_correct / total_gt)
        print("FP rate:", total_fp / vols)
        print("Total FPs:", total_fp)
        print("Total False Negatives:", f"{int(np.sum(fn_per_case))}/{total_gt}")
        # print(
        #     "FP rate (Healthy patients):",
        #     (
        #         None
        #         if len(healthy_case_indexes) == 0
        #         else np.array(fp_per_case)[healthy_case_indexes].sum()
        #         / len(healthy_case_indexes)
        #     ),
        # )
        # print(
        #     "Patient-level specificity",
        #     f"{len(healthy_case_indexes) - len(healthy_fp)}/{len(healthy_case_indexes)}",
        # )
        # print(
        #     "Patient-level sensitivity",
        #     f"{len(sick_case_indics) - sum(sick_fn > 0)} / {len(sick_case_indics)}",
        # )
        # for sz in ["small", "medium", "large"]:
        #     total_sz = df_labels[df_labels["size"] == sz].shape[0]
        #     if total_sz > 0:
        #         correct_sz = df_labels[
        #             (df_labels["size"] == sz) & (df_labels["detected"] == 1)
        #         ].shape[0]

        #         print(
        #             f"Recall {sz}: {100 * correct_sz / total_sz:.1f} ({correct_sz}/{total_sz})"
        #         )


------------------------------

Running iou_thr: 0.1 at 50k
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/inference_50k/predict_roi_jisoo_filtered.csv
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/wiwawe/iou0.1_froc_50k


/tmp/ipykernel_2397951/3703505683.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds["is_fp"] = np.array(fp_list).astype(np.int8)


Dataset: cta_rsna_ane, Model: decoder_only_no_rec_input_edt, Mode: base
Confidence threshold: 0.8
Total correct 940.0
FP rate: 1.841043890865955
Total FPs: 1552
Total False Negatives: 87/1027

------------------------------

Running iou_thr: 0.1 at 50k
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/inference_50k/predict_roi_jisoo_filtered.csv
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/wiwawe/iou0.1_froc_50k


/tmp/ipykernel_2397951/3703505683.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds["is_fp"] = np.array(fp_list).astype(np.int8)


Dataset: cta_rsna_ane, Model: decoder_only_no_rec_input_edt, Mode: 1
Confidence threshold: 0.8
Total correct 936.0
FP rate: 1.2194543297746145
Total FPs: 1028
Total False Negatives: 91/1027

------------------------------

Running iou_thr: 0.1 at 50k
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/inference_50k/predict_roi_jisoo_filtered.csv
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/wiwawe/iou0.1_froc_50k


...

/tmp/ipykernel_2397951/3703505683.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds["is_fp"] = np.array(fp_list).astype(np.int8)


Dataset: cta_rsna_ane, Model: decoder_only_no_rec_input_edt, Mode: 3
Confidence threshold: 0.8
Total correct 917.0
FP rate: 1.3973902728351126
Total FPs: 1178
Total False Negatives: 110/1027

------------------------------

Running iou_thr: 0.1 at 50k
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/inference_50k/predict_roi_jisoo_filtered.csv
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/wiwawe/iou0.1_froc_50k


/tmp/ipykernel_2397951/3703505683.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds["is_fp"] = np.array(fp_list).astype(np.int8)


Dataset: cta_rsna_ane, Model: decoder_only_no_rec_input_edt, Mode: 4
Confidence threshold: 0.8
Total correct 827.0
FP rate: 0.9478054567022538
Total FPs: 799
Total False Negatives: 200/1027

------------------------------

Running iou_thr: 0.1 at 50k
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/inference_50k/predict_roi_jisoo_filtered.csv
../../results/cta_rsna_ane/decoder_only_no_rec_input_edt/wiwawe/iou0.1_froc_50k
Dataset: cta_rsna_ane, Model: decoder_only_no_rec_input_edt, Mode: 5
Confidence threshold: 0.8
Total correct 914.0
FP rate: 1.0344009489916963
Total FPs: 872
Total False Negatives: 113/1027


/tmp/ipykernel_2397951/3703505683.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds["is_fp"] = np.array(fp_list).astype(np.int8)


In [57]:
preds["is_fp"].value_counts()

is_fp
0    146
1     72
Name: count, dtype: int64

In [58]:
# for each prediction in preds, check if it matches to any gt


def compute_iou(pred, gt):
    pred_x1 = pred[0] - pred[3] / 2
    pred_y1 = pred[1] - pred[4] / 2
    pred_z1 = pred[2] - pred[5] / 2
    pred_x2 = pred[0] + pred[3] / 2
    pred_y2 = pred[1] + pred[4] / 2
    pred_z2 = pred[2] + pred[5] / 2
    gt_x1 = gt[0] - gt[3] / 2
    gt_y1 = gt[1] - gt[4] / 2
    gt_z1 = gt[2] - gt[5] / 2
    gt_x2 = gt[0] + gt[3] / 2
    gt_y2 = gt[1] + gt[4] / 2
    gt_z2 = gt[2] + gt[5] / 2

    volume_pred = (pred_x2 - pred_x1) * (pred_y2 - pred_y1) * (pred_z2 - pred_z1)
    volume_gt = (gt_x2 - gt_x1) * (gt_y2 - gt_y1) * (gt_z2 - gt_z1)

    x_overlap = max(0, min(pred_x2, gt_x2) - max(pred_x1, gt_x1))
    y_overlap = max(0, min(pred_y2, gt_y2) - max(pred_y1, gt_y1))
    z_overlap = max(0, min(pred_z2, gt_z2) - max(pred_z1, gt_z1))

    intersection = x_overlap * y_overlap * z_overlap

    int_over_min = intersection / min(volume_pred, volume_gt)
    int_over_union = intersection / (volume_pred + volume_gt - intersection)  # not used
    return int_over_min


df_infundibulum = pd.read_csv(
    "/projects/vig/Datasets/aneurysm/cta_datasets/hospital140/annotations_infundibulum2.csv"
)

# check if anything in preds has iou over the specified iou threshold with stuff in df infundibulum

iou_thr_inf = 0.1
for i, pred in preds.iterrows():
    pred_seriesuid = pred["seriesuid"]
    pred_xyz = np.array(
        [
            pred["coordX"],
            pred["coordY"],
            pred["coordZ"],
            pred["w"],
            pred["h"],
            pred["d"],
        ]
    )
    for j, infundibulum in df_infundibulum.iterrows():
        infundibulum_seriesuid = infundibulum["seriesuid"]
        infundibulum_xyz = np.array(
            [
                infundibulum["coordX"],
                infundibulum["coordY"],
                infundibulum["coordZ"],
                infundibulum["w"],
                infundibulum["h"],
                infundibulum["d"],
            ]
        )
        if pred_seriesuid == infundibulum_seriesuid:
            iou = compute_iou(pred_xyz, infundibulum_xyz)
            if iou >= iou_thr_inf and pred["probability"] >= t:
                preds.at[i, "is_infundibulum"] = 1
preds[preds["is_infundibulum"] == True]

/tmp/ipykernel_1181437/3372228566.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds.at[i, "is_infundibulum"] = 1


,seriesuid,probability,coordZ,coordY,coordX,d,h,w,overlap,is_fp,is_infundibulum
361,CB_00009_0000.nii.gz,0.995782,156.94704,317.58840,303.24277,5.446918,8.719188,8.204179,1.0,1,1.0
1361,CB_00034_0000.nii.gz,0.998853,164.30022,324.37164,289.08370,5.647811,6.637800,6.387499,1.0,1,1.0
2120,CB_00053_0000.nii.gz,0.997276,159.92177,325.86620,247.80896,5.332250,6.963077,6.224815,1.0,1,1.0
3200,CB_00080_0000.nii.gz,0.999781,145.35536,350.15033,285.78100,10.650771,11.384172,17.702630,1.0,1,1.0
3201,CB_00080_0000.nii.gz,0.993052,149.76405,346.11000,294.05017,5.829138,7.395052,7.956746,1.0,1,1.0
3320,CB_00083_0000.nii.gz,0.999788,158.32088,318.72810,306.52826,18.443330,14.911891,16.658102,1.0,0,1.0


In [59]:
preds = preds[preds["is_infundibulum"] != 1]
preds["is_fp"].value_counts()

is_fp
0    145
1     67
Name: count, dtype: int64

In [60]:
df_labels[df_labels["detected"] == 0]["size"].value_counts()

size
small     43
medium    21
large      2
Name: count, dtype: int64

In [61]:
df_labels[df_labels["detected"] == 1]["size"].value_counts()

size
medium    99
small     34
large     13
Name: count, dtype: int64

In [114]:
(33)/(45+33)

0.4230769230769231

In [104]:
33/(33+44)

0.42857142857142855

In [62]:


iou_thr = 0.2

conf_thres = t
conf_thres_alt = 0.5

# preds = pd.read_csv(
#    "/home/azureuser/workspace/medical/deform-aneurysm-detection/results/internal_train/cnn_4l_input_edt_amp_ogopt/inference_final/predict.csv"
# )
# df_labels = pd.read_csv("/projects/vig/Datasets/aneurysm/cta_datasets/internal_train/annotations.csv")

# preds = pd.read_csv(path_preds)
df_labels = pd.read_csv(label_file)
preds = pd.read_csv(path_preds)
preds = preds[preds["overlap"] > 0.5]
preds = preds[preds["probability"] > conf_thres_alt]
df_labels = add_size_based_fields(df_labels)
df_labels = df_labels.sort_values(by="seriesuid")
df_labels["associated_confidence"] = 0.0
df_labels["detected"] = 0
preds["is_fp"] = 1


In [ ]:
for i, pred in preds.iterrows():
    pred_seriesuid = pred["seriesuid"]
    pred_xyz = np.array(
        [
            pred["coordX"],
            pred["coordY"],
            pred["coordZ"],
            pred["w"],
            pred["h"],
            pred["d"],
        ]
    )
    for j, gt in df_labels.iterrows():
        gt_seriesuid = gt["seriesuid"]
        gt_xyz = np.array(
            [gt["coordX"], gt["coordY"], gt["coordZ"], gt["w"], gt["h"], gt["d"]]
        )
        if pred_seriesuid == gt_seriesuid:
            iou = compute_iou(pred_xyz, gt_xyz)
            if iou >= iou_thr and pred["probability"] >= conf_thres:
                preds.at[i, "is_fp"] = 0
                df_labels.at[j, "detected"] = 1
                # if confidence is larger than the current confidence, update it
                if pred["probability"] > df_labels.at[j, "associated_confidence"]:
                    df_labels.at[j, "associated_confidence"] = pred["probability"]
            elif iou >= iou_thr and pred["probability"] < conf_thres:
                if pred["probability"] > df_labels.at[j, "associated_confidence"]:
                    df_labels.at[j, "associated_confidence"] = pred["probability"]
preds = preds[preds["probability"] > conf_thres]

In [64]:
df_labels.value_counts("detected", normalize=True)

detected
1    0.707547
0    0.292453
Name: proportion, dtype: float64

In [102]:
df_labels[df_labels["detected"] == 0]["size"].value_counts()

size
small     34
medium    11
Name: count, dtype: int64

In [22]:
preds["is_fp"].value_counts()

is_fp
0    166
1     93
Name: count, dtype: int64

In [16]:
preds["is_fp"].value_counts()

is_fp
0    163
1     54
Name: count, dtype: int64

In [36]:
len(df_infundibulum)

10

In [26]:
192 / 140

1.3714285714285714

219

In [ ]:
df_smol_suc = df_labels[(df_labels["size"] == "small") & (df_labels["detected"] == 1)]
df_smol_fail = df_labels[(df_labels["size"] == "small") & (df_labels["detected"] == 0)]

sizes_suc = df_smol_suc["diameter"].values
sizes_fail = df_smol_fail["diameter"].values

import matplotlib.pyplot as plt
import seaborn as sns 

plt.figure(figsize=(8,5))
plt.hist(sizes_suc, bins=5, density=False, alpha=0.5, label="Detected")
plt.hist(sizes_fail, bins=5, density=False, alpha=0.5, label="Missed")
#sns.histplot(sizes_suc, label="Detected", stat="
#sns.kdeplot(sizes_suc, label="Detected", fill=True, common_norm = False )
#sns.kdeplot(sizes_fail, label="Missed", fill=True, common_norm = False)
plt.xlabel("Aneurysm size (mm)")
plt.ylabel("Count")
plt.title("Detection of small aneurysms by size (augmented model)")
plt.xlim(0,5)
plt.ylim(0,20)
plt.legend()
